In [1]:
import numpy as np
import matplotlib.pyplot as plt


def infer_box2_from_box1(box1_coords, pattern_height):
    """
    Infer box2 coordinates by reflecting box1 across the x-axis (horizontal flip).
    
    Reflection across x-axis (center of pattern): y -> (pattern_height - y)
    So box coordinates ((y_min, y_max), (x_min, x_max)) get y-flipped around center
    
    Parameters:
    -----------
    box1_coords : tuple of tuples
        Coordinates for first box: ((y_min, y_max), (x_min, x_max))
    pattern_height : int
        Height of the diffraction pattern (to calculate center for reflection)
    
    Returns:
    --------
    tuple : Reflected box2 coordinates ((y_min, y_max), (x_min, x_max))
    """
    (y_min, y_max), (x_min, x_max) = box1_coords
    # Reflection across x-axis (horizontal center line)

    # Flip y coordinates around the center    return box2_coords

    box2_y_min = pattern_height - y_max    
    box2_y_max = pattern_height - y_min
    box2_coords = ((box2_y_min, box2_y_max), (x_min, x_max))
    return box2_coords

In [2]:
def calculate_box_asymmetry_score(diffraction_pattern, box1_coords, box2_coords):
    """
    Calculate asymmetry score using masked center of mass approach.
    
    Instead of summing intensities from the boxes, this function creates a binary mask
    from the union of both boxes and computes the weighted center of mass. The
    y-displacement from the pattern center is used as the asymmetry metric.
    
    Parameters:
    -----------
    diffraction_pattern : ndarray or abtem measurement
        2D diffraction pattern
    box1_coords : tuple of tuples
        Coordinates for first box: ((y_min, y_max), (x_min, x_max))
    box2_coords : tuple of tuples
        Coordinates for second box: ((y_min, y_max), (x_min, x_max))
    
    Returns:
    --------
    dict : Dictionary containing asymmetry metrics (now based on COM displacement)
    """
    # Convert to numpy array
    if hasattr(diffraction_pattern, 'array'):
        pattern = np.array(diffraction_pattern.array)
    else:
        pattern = np.array(diffraction_pattern)

    pattern = np.asarray(pattern, dtype=np.float64)

    # Ensure 2D
    if pattern.ndim > 2:
        pattern = np.squeeze(pattern)

    ny, nx = pattern.shape

    # Helper function to convert coordinates
    def parse_coords(coords, ny, nx):
        (y_min, y_max), (x_min, x_max) = coords
        
        # If fractional (0-1), convert to pixels
        if 0 <= y_min <= 1 and 0 <= y_max <= 1:
            y_min = int(y_min * ny)
            y_max = int(y_max * ny)
        if 0 <= x_min <= 1 and 0 <= x_max <= 1:
            x_min = int(x_min * nx)
            x_max = int(x_max * nx)
        
        # Ensure within bounds
        y_min = max(0, min(int(y_min), ny-1))
        y_max = max(0, min(int(y_max), ny))
        x_min = max(0, min(int(x_min), nx-1))
        x_max = max(0, min(int(x_max), nx))
        
        return y_min, y_max, x_min, x_max
    
    # Parse box coordinates
    y1_min, y1_max, x1_min, x1_max = parse_coords(box1_coords, ny, nx)
    y2_min, y2_max, x2_min, x2_max = parse_coords(box2_coords, ny, nx)
    
    # Create separate binary masks for each box
    mask1 = np.zeros((ny, nx), dtype=bool)
    mask2 = np.zeros((ny, nx), dtype=bool)
    
    mask1[y1_min:y1_max, x1_min:x1_max] = True
    mask2[y2_min:y2_max, x2_min:x2_max] = True

    # Calculate center of mass for each box separately using scipy's method
    from scipy.ndimage import center_of_mass
    
    mask_1_com = center_of_mass(pattern * mask1)
    mask_2_com = center_of_mass(pattern * mask2)
    
    # Calculate midpoint of the two COMs
    midpoint_y = (mask_1_com[0] + mask_2_com[0]) / 2
    midpoint_x = (mask_1_com[1] + mask_2_com[1]) / 2
    
    # Calculate displacement from center
    center_y, center_x = ny / 2, nx / 2
    dy = midpoint_y - center_y
    dx = midpoint_x - center_x
    
    # COM y-displacement as a percentage of pattern height
    # Positive = midpoint above center, negative = below center
    com_y_displacement = -(dy / ny) * 100

    # Extract box regions for additional info
    box1_region = pattern[y1_min:y1_max, x1_min:x1_max]
    box2_region = pattern[y2_min:y2_max, x2_min:x2_max]

    box1_intensity = np.sum(box1_region)
    box2_intensity = np.sum(box2_region)
    box1_mean = np.mean(box1_region)
    box2_mean = np.mean(box2_region)
    total_intensity = box1_intensity + box2_intensity
    intensity_asymmetry = box1_intensity - box2_intensity

    return {
        'com_y_displacement': com_y_displacement,
        'intensity_asymmetry': intensity_asymmetry,
        'y_displacement': dy,
        'x_displacement': dx,
        'midpoint_y': midpoint_y,
        'midpoint_x': midpoint_x,
        'box1_com_y': mask_1_com[0],
        'box1_com_x': mask_1_com[1],
        'box2_com_y': mask_2_com[0],
        'box2_com_x': mask_2_com[1],
        'total_intensity': total_intensity,
        'box1_intensity': box1_intensity,
        'box2_intensity': box2_intensity,
        'box1_mean': box1_mean,
        'box2_mean': box2_mean,
        'box1_coords': (y1_min, y1_max, x1_min, x1_max),
        'box2_coords': (y2_min, y2_max, x2_min, x2_max)
    }

In [3]:
def process_material_data(numpy_file, box_config, metric='intensity_asymmetry', sign_flip=False, tilt_flip=False):
    """
    Process diffraction data for a single material.
    
    Parameters:
    -----------
    numpy_file : str
        Path to the .npz file containing diffraction data
    box_config : dict
        Box configuration with 'box1' and 'box2' keys containing (y_min, y_max, x_min, x_max) tuples,
        or None to use boxes from file
    metric : str
        Either 'intensity_asymmetry' or 'com_y_displacement'
    sign_flip : bool
        If True, multiply the metric by -1
    tilt_flip : bool
        If True, flip the sign of tilt_y_values
        
    Returns:
    --------
    dict : Dictionary containing tilt_y_values and metric values
    """
    data = np.load(numpy_file)
    diffraction = data['diffraction_images']
    
    # Use custom box config or load from file
    if box_config is None:
        box1 = data['horizontal_boxes_box1_coords'][0][0]
        box2 = data['horizontal_boxes_box2_coords'][0][0]
        box1_coords = ((box1[0], box1[1]), (box1[2], box1[3]))
        box2_coords = ((box2[0], box2[1]), (box2[2], box2[3]))
    else:
        box1_coords = box_config['box1']
        box2_coords = box_config['box2']
    
    # Calculate asymmetry scores
    metric_values = []
    for i in range(diffraction.shape[0]):
        asymmetry_score = calculate_box_asymmetry_score(
            diffraction[i][0], box1_coords, box2_coords
        )
        value = asymmetry_score[metric]
        if sign_flip:
            value = -value
        metric_values.append(value)
    
    # Get tilt values
    tilt_y = data['tilt_y_values']
    if tilt_flip:
        tilt_y = [-i for i in tilt_y]
    
    return {
        'tilt_y': tilt_y,
        'metric': metric_values,
        'data': data,
        'box1_coords': box1_coords,
        'box2_coords': box2_coords
    }

In [4]:
# ==============================================================================
# CONFIGURATION: Intensity Box Analysis
# ==============================================================================

# File paths

INTENSITY_FILES = {
    # 'PTO': "../asymmetry_sweeps/PTO_tiltX0.0to0.0_tiltY-0.1to0.1_81pts_sc20x20x154_det35_box_phonons10_20260306_144509/asymmetry_data.npz",
    'PTO': "kikuchi_simulations/PTO",
    # 'BTO': "../fintal_results/BTO_tiltX0.0to0.0_tiltY-0.1to0.1_41pts_sc20x20x154_det35_box_phonons20_20260306_063214/asymmetry_data.npz",
    'BTO': "kikuchi_simulations/BTO",
    'Boracite': "kikuchi_simulations/boracite"
}

# ------------------------------------------------------------------------------
# Box pair colour palette — edit here to change colours everywhere.
# Each pair uses ONE colour: box1 drawn solid, box2 drawn dashed (same colour).
# All chosen to contrast against the inferno colormap (blacks, purples, oranges).
# ------------------------------------------------------------------------------
C_200_RANGE  = '#d31f11'   # cyan
C_300_RANGE  = '#f47a00'   # magenta
C_200        = '#62c8d3'   # chartreuse
C_300        = '#007191'   # turquoise

# C_200_RANGE  = '#082a54'   # cyan
# C_300_RANGE  = '#e02b35'   # magenta
# C_200        = '#f0c571'   # chartreuse
# C_300        = '#59a89c'   # turquoise

# C_200_RANGE  = '#0000a2'   # cyan
# C_300_RANGE  = '#bc272d'   # magenta
# C_200        = '#e9c716'   # chartreuse
# C_300        = '#50ad9f'   # turquoise

# C_200_RANGE  = '#4a2377'   # cyan
# C_300_RANGE  = '#8cc5e3'   # magenta
# C_200        = '#0d7d87'   # chartreuse
# C_300        = '#f55f74'   # turquoise

# ==============================================================================
# CONFIGURATION: Kikuchi COM Analysis
# ==============================================================================

# Directories containing batch01…batch10 sub-folders (one frozen-phonon run each)
KIKUCHI_FILES = {
    'PTO':      'kikuchi_simulations/PTO',
    'BTO':      'kikuchi_simulations/BTO',
    'Boracite': 'kikuchi_simulations/boracite',
}

# ------------------------------------------------------------------------------
# Kikuchi colour palette — reuses the same C_* variables defined in the
# Intensity config cell so colours are consistent across both analyses.
# ------------------------------------------------------------------------------
C_KIKUCHI_DEFAULT    = C_200_RANGE   # cyan   — same as [200] range
C_KIKUCHI_TOP_THIRD  = C_300_RANGE   # magenta
C_KIKUCHI_TOP_SIXTH  = C_200         # chartreuse

KIKUCHI_BOX_CONFIGS = {
    'PTO': [
        {
            'name': 'Default',
            'box': ((200, 350), (497, 527)),
            'color': C_KIKUCHI_DEFAULT,
        },
        {
            'name': 'top third',
            'box': ((200, 250), (497, 527)),
            'color': C_KIKUCHI_TOP_THIRD,
        },
        {
            'name': 'bottom third',
            'box': ((300, 350), (497, 527)),
            'color': C_300,
        },
        {
            'name': 'middle third',
            'box': ((250, 300), (497, 527)),
            'color': C_KIKUCHI_TOP_SIXTH,
        },
        {
            'name': 'top sixth',
            'box': ((200, 215), (497, 527)),
            'color': C_KIKUCHI_TOP_SIXTH,
        },
    ],

    'BTO': [
        {
            'name': 'Default',
            'box': ((200, 350), (497, 527)),
            'color': C_KIKUCHI_DEFAULT,
        },
        {
            'name': 'top third',
            'box': ((200, 250), (497, 527)),
            'color': C_KIKUCHI_TOP_THIRD,
        },
        {
            'name': 'bottom third',
            'box': ((300, 350), (497, 527)),
            'color': C_300,
        },
        {
            'name': 'middle third',
            'box': ((250, 300), (497, 527)),
            'color': C_KIKUCHI_TOP_SIXTH,
        },
        {
            'name': 'top sixth',
            'box': ((200, 215), (497, 527)),
            'color': C_KIKUCHI_TOP_SIXTH,
        },
    ],

    'Boracite': [
        {
            'name': 'Default',
            'box': ((200, 350), (497, 527)),
            'color': C_KIKUCHI_DEFAULT,
        },
        {
            'name': 'top third',
            'box': ((200, 250), (497, 527)),
            'color': C_KIKUCHI_TOP_THIRD,
        },
        {
            'name': 'bottom third',
            'box': ((300, 350), (497, 527)),
            'color': C_300,
        },
        {
            'name': 'middle third',
            'box': ((250, 300), (497, 527)),
            'color': C_KIKUCHI_TOP_SIXTH,
        },
        {
            'name': 'top sixth',
            'box': ((200, 215), (497, 527)),
            'color': C_KIKUCHI_TOP_SIXTH,
        },
    ],
}

# Processing options
KIKUCHI_PROCESSING = {
    'PTO': {'sign_flip': False, 'tilt_flip': True},
    'BTO': {'sign_flip': False, 'tilt_flip': True},
    'Boracite': {'sign_flip': False, 'tilt_flip': True}
}


In [5]:
def visualize_pattern_with_boxes(material, data_files, box_configs, pattern_idx=0,
                                   custom_box1=None, custom_box2=None, ax=None):
    """
    Visualize a diffraction pattern with box overlays. Supports multiple box pairs.
    Box2 is automatically inferred from box1 by x-axis reflection unless explicitly provided.

    Each box pair uses ONE colour (box1 solid, box2 dashed) so the reflected pair
    is clearly identified as the same measurement region, not a different one.
    """
    from matplotlib.patches import Rectangle
    from matplotlib.colors import PowerNorm

    # Load data
    data = np.load(data_files[material])
    pattern = data['diffraction_images'][pattern_idx][0]

    # if material == "PTO":
    #     pattern = np.transpose(pattern)

    tilt_y = data['tilt_y_values'][pattern_idx]
    pattern_height = pattern.shape[0]

    # Fallback colours when no colour is specified in config
    _fallback_colors = [C_200_RANGE, C_300_RANGE, C_200, C_300, '#FF69B4']

    def _resolve_color(box_pair, idx):
        """Return a single colour string for a box pair."""
        c = box_pair.get('color', None)
        if c is None:
            return _fallback_colors[idx % len(_fallback_colors)]
        return c[0] if isinstance(c, list) else c

    # Build list of box pairs to plot
    box_pairs = []

    if custom_box1 is not None:
        if custom_box2 is None:
            custom_box2 = infer_box2_from_box1(custom_box1, pattern_height)
        box_pairs = [{'name': 'Custom', 'box1': custom_box1, 'box2': custom_box2,
                      'color': _fallback_colors[0]}]
    elif box_configs[material] is not None:
        config = box_configs[material]
        if isinstance(config, list):
            for item in config:
                if 'box' in item:
                    box1 = item['box']
                    box2 = infer_box2_from_box1(box1, pattern_height)
                    box_pairs.append({'name': item.get('name', 'Unnamed'),
                                      'box1': box1, 'box2': box2,
                                      'color': item.get('color', None)})
                else:
                    box_pairs.append(item)
        else:
            if 'box' in config:
                box1, box2 = config['box'], infer_box2_from_box1(config['box'], pattern_height)
            else:
                box1, box2 = config['box1'], config['box2']
            box_pairs = [{'name': 'Default', 'box1': box1, 'box2': box2,
                          'color': config.get('color', None)}]
    else:
        box1 = data['horizontal_boxes_box1_coords'][0][0]
        box2 = data['horizontal_boxes_box2_coords'][0][0]
        box1_coords = ((box1[0], box1[1]), (box1[2], box1[3]))
        box2_coords = ((box2[0], box2[1]), (box2[2], box2[3]))
        box_pairs = [{'name': 'File', 'box1': box1_coords, 'box2': box2_coords,
                      'color': _fallback_colors[0]}]

    # Display pattern
    vmin = np.percentile(pattern, PERCENTILE_MIN)
    vmax = np.percentile(pattern, PERCENTILE_MAX)

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 10))

    im = ax.imshow(pattern, cmap='inferno', origin='lower',
                   norm=PowerNorm(gamma=GAMMA, vmin=vmin, vmax=vmax))

    # Add box overlays
    all_asymmetry_scores = []
    for idx, box_pair in enumerate(box_pairs):
        box1_coords = box_pair['box1']
        box2_coords = box_pair['box2']
        pair_name = box_pair.get('name', f'Pair {idx+1}')
        color = _resolve_color(box_pair, idx)

        (y1_min, y1_max), (x1_min, x1_max) = box1_coords
        (y2_min, y2_max), (x2_min, x2_max) = box2_coords

        # Both boxes share the same colour; line style distinguishes box1 from box2
        rect1 = Rectangle((x1_min, y1_min), x1_max - x1_min, y1_max - y1_min,
                           linewidth=2, edgecolor=color, facecolor='none',
                           label=f'{pair_name} - Box 1', linestyle='-')
        rect2 = Rectangle((x2_min, y2_min), x2_max - x2_min, y2_max - y2_min,
                           linewidth=2, edgecolor=color, facecolor='none',
                           label=f'{pair_name} - Box 2', linestyle='--')
        ax.add_patch(rect1)
        ax.add_patch(rect2)
        ax.set_xticks([])
        ax.set_yticks([])

        asym_score = calculate_box_asymmetry_score(pattern, box1_coords, box2_coords)
        asym_score['name'] = pair_name
        all_asymmetry_scores.append(asym_score)

        print(f"\n{material} - {pair_name} Box Coordinates:")
        print(f"  Box 1: ((y_min={y1_min}, y_max={y1_max}), (x_min={x1_min}, x_max={x1_max}))")
        print(f"  Box 2: ((y_min={y2_min}, y_max={y2_max}), (x_min={x2_min}, x_max={x2_max})) [auto-inferred]")
        print(f"  Intensity Asymmetry: {asym_score['intensity_asymmetry']:.2f}")
        print(f"  COM Y-Disp: {asym_score['com_y_displacement']:.3f}%")

    # Centre crosshair
    center_y, center_x = pattern.shape[0] / 2, pattern.shape[1] / 2
    ax.axhline(center_y, color='white', linestyle='--', alpha=0.3, linewidth=1)
    ax.axvline(center_x, color='white', linestyle='--', alpha=0.3, linewidth=1)

    title = f'{material} - Pattern {pattern_idx}'
    if len(all_asymmetry_scores) == 1:
        title += (f'\nIntensity Asym: {all_asymmetry_scores[0]["intensity_asymmetry"]:.2f}, '
                  f'COM Y-Disp: {all_asymmetry_scores[0]["com_y_displacement"]:.3f}%')
    else:
        title += f'\n{len(all_asymmetry_scores)} box pairs shown'

    ax.set_title(title, fontsize=12, pad=10)
    ax.legend(loc='upper right', fontsize=8)

    print(f"Pattern shape: {pattern.shape}")

    if ax is None:
        plt.tight_layout()
        plt.show()

    print(f"Tilt Y: {tilt_y}")

    return all_asymmetry_scores if len(all_asymmetry_scores) > 1 else all_asymmetry_scores[0]

In [6]:
def _parse_box_coords(box_coords, H, W):
    """Convert box coords (fractional or pixel) to integer pixel indices."""
    (y_min, y_max), (x_min, x_max) = box_coords
    if 0 <= y_min <= 1 and 0 <= y_max <= 1:
        y_min, y_max = int(y_min * H), int(y_max * H)
    if 0 <= x_min <= 1 and 0 <= x_max <= 1:
        x_min, x_max = int(x_min * W), int(x_max * W)
    y_min = max(0, min(int(y_min), H - 1))
    y_max = max(0, min(int(y_max), H))
    x_min = max(0, min(int(x_min), W - 1))
    x_max = max(0, min(int(x_max), W))
    return y_min, y_max, x_min, x_max


def _com_y_displacement_vectorized(patterns, box1_coords, box2_coords, sign_flip=False):
    """
    Vectorized COM y-displacement over all tilt points.

    patterns : ndarray, shape (n_tilt, H, W), float64
    Returns  : ndarray, shape (n_tilt,)
    """
    n_tilt, H, W = patterns.shape
    y1_min, y1_max, x1_min, x1_max = _parse_box_coords(box1_coords, H, W)
    y2_min, y2_max, x2_min, x2_max = _parse_box_coords(box2_coords, H, W)

    # Slice box regions: (n_tilt, box_h, box_w)
    b1 = patterns[:, y1_min:y1_max, x1_min:x1_max]
    b2 = patterns[:, y2_min:y2_max, x2_min:x2_max]

    # Weighted y-coordinate sum for COM (no full-pattern masking needed)
    ys1 = np.arange(y1_min, y1_max, dtype=np.float64)  # (box1_h,)
    ys2 = np.arange(y2_min, y2_max, dtype=np.float64)  # (box2_h,)

    # einsum: sum over y and x, weighted by ys  →  (n_tilt,)
    com_y1 = np.einsum('tyx,y->t', b1, ys1) / b1.sum(axis=(1, 2))
    com_y2 = np.einsum('tyx,y->t', b2, ys2) / b2.sum(axis=(1, 2))

    midpoint_y = (com_y1 + com_y2) / 2
    dy = midpoint_y - H / 2
    result = -(dy / H) * 100
    return -result if sign_flip else result


def _intensity_asymmetry_vectorized(patterns, box1_coords, box2_coords, sign_flip=False):
    """Vectorized intensity asymmetry (box1_sum - box2_sum) over all tilt points."""
    n_tilt, H, W = patterns.shape
    y1_min, y1_max, x1_min, x1_max = _parse_box_coords(box1_coords, H, W)
    y2_min, y2_max, x2_min, x2_max = _parse_box_coords(box2_coords, H, W)

    b1_sum = patterns[:, y1_min:y1_max, x1_min:x1_max].sum(axis=(1, 2))
    b2_sum = patterns[:, y2_min:y2_max, x2_min:x2_max].sum(axis=(1, 2))
    result = b1_sum - b2_sum
    return -result if sign_flip else result


def process_material_data_multi_boxes(numpy_file, box_configs, metric='intensity_asymmetry',
                                      sign_flip=False, tilt_flip=False, material=None):
    """
    Process diffraction data for a single material with multiple box pairs.
    Box2 is automatically inferred from box1 by x-axis reflection unless explicitly provided.
    All tilt points are processed at once using vectorized numpy operations (no per-frame loop).
    """
    data = np.load(numpy_file)
    # shape: (n_tilt, 1, H, W)
    diffraction = data['diffraction_images']
    patterns = diffraction[:, 0].astype(np.float64)  # (n_tilt, H, W)
    n_tilt, H, W = patterns.shape

    # Determine box pairs
    if box_configs is None:
        box1 = data['horizontal_boxes_box1_coords'][0][0]
        box2 = data['horizontal_boxes_box2_coords'][0][0]
        box_pairs = [{'name': 'Default',
                      'box1': ((box1[0], box1[1]), (box1[2], box1[3])),
                      'box2': ((box2[0], box2[1]), (box2[2], box2[3]))}]
    elif isinstance(box_configs, list):
        box_pairs = []
        for item in box_configs:
            if 'box' in item:
                box1 = item['box']
                box2 = infer_box2_from_box1(box1, H)
                box_pairs.append({'name': item.get('name', 'Unnamed'), 'box1': box1, 'box2': box2})
            else:
                box_pairs.append(item)
    else:
        if 'box' in box_configs:
            box1 = box_configs['box']
            box2 = infer_box2_from_box1(box1, H)
        else:
            box1, box2 = box_configs['box1'], box_configs['box2']
        box_pairs = [{'name': 'Default', 'box1': box1, 'box2': box2}]

    # Tilt values
    tilt_y = data['tilt_y_values']
    if tilt_flip:
        tilt_y = [-v for v in tilt_y]

    # Choose vectorized metric function
    if metric == 'com_y_displacement':
        _compute = _com_y_displacement_vectorized
    elif metric == 'intensity_asymmetry':
        _compute = _intensity_asymmetry_vectorized
    else:
        raise ValueError(f"Unknown metric: {metric!r}")

    results = {'tilt_y': tilt_y, 'box_pairs': {}}

    for box_pair in box_pairs:
        name = box_pair.get('name', 'Unnamed')
        box1_coords = box_pair['box1']
        box2_coords = box_pair['box2']

        metric_values = _compute(patterns, box1_coords, box2_coords, sign_flip=sign_flip).tolist()

        results['box_pairs'][name] = {
            'metric': metric_values,
            'box1_coords': box1_coords,
            'box2_coords': box2_coords,
        }

    # Backward compatibility: single pair → also expose at top level
    if len(box_pairs) == 1:
        only = box_pairs[0].get('name', 'Default')
        results['metric'] = results['box_pairs'][only]['metric']
        results['box1_coords'] = box_pairs[0]['box1']
        results['box2_coords'] = box_pairs[0]['box2']

    results['data'] = data
    return results


In [7]:
import glob
import os

def process_kikuchi_multi_runs(material_dir, box_configs, metric='com_y_displacement',
                                sign_flip=False, tilt_flip=False, material=None):
    """
    Load all frozen-phonon batch runs from a directory and return mean ± std
    across runs for each box pair.

    Expects the directory layout:
        material_dir/
            {material}_asymmetry_sweeps_batch01/
                {run_folder}/
                    asymmetry_data.npz
            {material}_asymmetry_sweeps_batch02/
                ...

    Parameters
    ----------
    material_dir : str
        Path to the material directory containing batch sub-folders.
    box_configs : list of dict
        Box configurations passed straight through to process_material_data_multi_boxes.
    metric : str
        Metric key to extract ('com_y_displacement' or 'intensity_asymmetry').
    sign_flip, tilt_flip : bool
        Passed through to process_material_data_multi_boxes.
    material : str or None
        Material name, passed through for any material-specific logic.

    Returns
    -------
    dict with keys:
        'tilt_y'     – tilt values (from run 0)
        'box_pairs'  – dict of pair_name → {'mean', 'std', 'all_runs', ...}
        'n_runs'     – number of runs found
        'first_npz'  – path to first npz (use for pattern visualisation)
    """
    npz_files = sorted(
        glob.glob(os.path.join(material_dir, '*', '*', 'asymmetry_data.npz'))
    )
    if not npz_files:
        raise FileNotFoundError(f"No asymmetry_data.npz files found under {material_dir}")
    print(f"  {material}: found {len(npz_files)} runs")

    all_run_results = [
        process_material_data_multi_boxes(
            f, box_configs, metric=metric,
            sign_flip=sign_flip, tilt_flip=tilt_flip, material=material
        )
        for f in npz_files
    ]

    tilt_y = all_run_results[0]['tilt_y']
    pair_names = list(all_run_results[0]['box_pairs'].keys())

    aggregated = {
        'tilt_y': tilt_y,
        'box_pairs': {},
        'n_runs': len(npz_files),
        'first_npz': npz_files[0],
    }

    for pair_name in pair_names:
        # shape: (n_runs, n_tilt_points)
        all_metrics = np.array([
            r['box_pairs'][pair_name]['metric'] for r in all_run_results
        ])
        aggregated['box_pairs'][pair_name] = {
            'mean':       np.mean(all_metrics, axis=0),
            'std':        np.std(all_metrics, axis=0),
            'all_runs':   all_metrics,
            'box1_coords': all_run_results[0]['box_pairs'][pair_name]['box1_coords'],
            'box2_coords': all_run_results[0]['box_pairs'][pair_name]['box2_coords'],
        }

    return aggregated


In [8]:
# Process all materials for Kikuchi COM — aggregate across all frozen-phonon batch runs
kikuchi_results = {}

for material in ['PTO', 'BTO', 'Boracite']:
    print(f"Loading {material}...")
    result = process_kikuchi_multi_runs(
        KIKUCHI_FILES[material],
        KIKUCHI_BOX_CONFIGS[material],
        metric='com_y_displacement',
        material=material,
        **KIKUCHI_PROCESSING[material]
    )
    kikuchi_results[material] = result

    for pair_name, pair_data in result['box_pairs'].items():
        print(f"  [{pair_name}]  mean range [{pair_data['mean'].min():.3f}, "
              f"{pair_data['mean'].max():.3f}],  max std = {pair_data['std'].max():.3f}")


Loading PTO...
  PTO: found 10 runs


KeyboardInterrupt: 

In [ ]:
# Combined figure: diffraction patterns with box overlays (top) + Kikuchi COM plots (bottom)
# COM plots show mean line ± 1 std shaded band, aggregated over all frozen-phonon runs.
plt.style.use('seaborn-v0_8-talk')
fig, axes = plt.subplots(2, 3, figsize=(18, 11),
                         gridspec_kw={'height_ratios': [1.3, 1]})

material_settings = {
    'PTO':      {'marker': 'o', 'label': 'PbTiO₃'},
    'BTO':      {'marker': 's', 'label': 'BaTiO₃'},
    'Boracite': {'marker': '^', 'label': 'Cu₃B₇O₁₃Cl'},
}
box_pair_styles = ['-', '--', '-.', ':']
materials = ['PTO', 'BTO', 'Boracite']

# ── Top row: diffraction patterns at zero tilt ─────────────────────────────
# Load pattern from first batch run of each material for visualisation
kikuchi_first_files = {m: kikuchi_results[m]['first_npz'] for m in materials}

for col, material in enumerate(materials):
    data_tmp = np.load(kikuchi_first_files[material])
    zero_tilt_idx = len(data_tmp['tilt_y_values']) // 2
    visualize_pattern_with_boxes(material, kikuchi_first_files, KIKUCHI_BOX_CONFIGS,
                                 pattern_idx=zero_tilt_idx, ax=axes[0, col])
    axes[0, col].set_title(material_settings[material]['label'], fontsize=14, pad=6)

# ── Bottom row: Kikuchi COM mean ± std plots ───────────────────────────────
for col, material in enumerate(materials):
    ax = axes[1, col]
    settings = material_settings[material]
    result = kikuchi_results[material]
    n_runs = result['n_runs']

    pair_colors = {
        item['name']: (item['color'][0] if isinstance(item['color'], list) else item['color'])
        for item in KIKUCHI_BOX_CONFIGS[material]
        if 'name' in item and 'color' in item
    }

    tilt_y = np.array(result['tilt_y'])

    for idx, (pair_name, pair_data) in enumerate(result['box_pairs'].items()):
        color = pair_colors.get(pair_name, '#888888')
        mean = pair_data['mean']
        std  = pair_data['std']
        style = box_pair_styles[idx % len(box_pair_styles)]

        # Shaded ±1 std band
        ax.fill_between(tilt_y, mean - std, mean + std,
                        color=color, alpha=0.2)

        # Mean line
        ax.plot(tilt_y, mean,
                linestyle=style, marker=settings['marker'], color=color,
                linewidth=2.5, markersize=5, markerfacecolor=color,
                markeredgewidth=1.2, markeredgecolor='white',
                label=f'{pair_name} (n={n_runs})', alpha=0.9)

    ax.axhline(0, color='gray', linestyle='--', alpha=0.5, linewidth=1.5)
    ax.set_xlabel('Tilt Y (degrees)', fontsize=12)
    ax.set_ylabel('COM Y-Displacement (%)', fontsize=12)
    ax.legend(fontsize=9, frameon=True, loc='best')
    ax.grid(True, alpha=0.3, linestyle=':')

plt.suptitle('Kikuchi COM Y-Displacement Analysis', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()


# Load in the std plots 

In [ ]:
INPUT_NPZ = "kikuchi_com_analysis_results.npz"
MATERIALS = ['PTO', 'BTO', 'Boracite']

data = np.load(INPUT_NPZ)
print(f"Loaded {INPUT_NPZ} — {len(data.files)} arrays")

plt.style.use('seaborn-v0_8-talk')
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

for col, material in enumerate(MATERIALS):
    ax     = axes[col]
    meta   = material_settings[material]
    tilt_y = data[f'{material}_tilt_y']
    n_runs = int(data[f'{material}_n_runs'])

    for bp in KIKUCHI_BOX_CONFIGS:
        if bp == "Default":
            mean_key = f"{material}_{bp['key']}_mean"
            std_key  = f"{material}_{bp['key']}_std"
            if mean_key not in data.files:
                continue

            mean = data[mean_key]
            std  = data[std_key]
            color = bp['color']
            ls    = bp['ls']

            # ±1 std shaded band
            ax.fill_between(tilt_y, mean - std, mean + std,
                            color=color, alpha=0.20)

            # Mean line with markers
            ax.plot(tilt_y, mean,
                    linestyle=ls, marker=meta['marker'],
                    color=color, linewidth=2.5, markersize=5,
                    markerfacecolor=color, markeredgewidth=1.2,
                    markeredgecolor='white', alpha=0.9,
                    label=f"{bp['label']} (n={n_runs})")

    ax.axhline(0, color='gray', linestyle='--', alpha=0.5, linewidth=1.5)
    ax.set_title(meta['label'], fontsize=14)
    ax.set_xlabel('Tilt Y (degrees)', fontsize=12)
    ax.set_ylabel('COM Y-Displacement (%)', fontsize=12)
    ax.legend(fontsize=8, frameon=True, loc='best')
    ax.grid(True, alpha=0.3, linestyle=':')

plt.suptitle('Kikuchi COM Y-Displacement — mean ± 1 std across frozen-phonon runs',
                fontsize=14, y=1.02)
plt.tight_layout()

['PTO_tilt_y', 'PTO_n_runs', 'PTO_Default_mean', 'PTO_Default_std', 'PTO_Default_all_runs', 'PTO_top_third_mean', 'PTO_top_third_std', 'PTO_top_third_all_runs', 'PTO_bottom_third_mean', 'PTO_bottom_third_std', 'PTO_bottom_third_all_runs', 'PTO_middle_third_mean', 'PTO_middle_third_std', 'PTO_middle_third_all_runs', 'PTO_top_sixth_mean', 'PTO_top_sixth_std', 'PTO_top_sixth_all_runs', 'BTO_tilt_y', 'BTO_n_runs', 'BTO_Default_mean', 'BTO_Default_std', 'BTO_Default_all_runs', 'BTO_top_third_mean', 'BTO_top_third_std', 'BTO_top_third_all_runs', 'BTO_bottom_third_mean', 'BTO_bottom_third_std', 'BTO_bottom_third_all_runs', 'BTO_middle_third_mean', 'BTO_middle_third_std', 'BTO_middle_third_all_runs', 'BTO_top_sixth_mean', 'BTO_top_sixth_std', 'BTO_top_sixth_all_runs', 'Boracite_tilt_y', 'Boracite_n_runs', 'Boracite_Default_mean', 'Boracite_Default_std', 'Boracite_Default_all_runs', 'Boracite_top_third_mean', 'Boracite_top_third_std', 'Boracite_top_third_all_runs', 'Boracite_bottom_third_mean',